# 09 – Model Evaluation & Comparison

**Pipeline stage:** Tổng hợp & Đánh giá Model

Notebook này nhằm mục đích tổng hợp, so sánh hiệu suất dự báo và khả năng giải thích (đặc biệt là $R^2$) từ các phương pháp tiếp cận khác nhau:
1. **Các mô hình Gravity Cơ bản (FEM, REM, PPML)** (từ Notebook 04 và 05): Đánh giá các biến số cơ bản của mô hình lực hấp dẫn.
2. **Mô hình Kinh tế lượng Không gian (SDM / SLX)** (từ Notebook 07): Tập trung vào khả năng giải thích hiệu ứng lan tỏa không gian.
3. **Các mô hình Machine Learning (RF, XGBoost, LightGBM)** (từ Notebook 08): Tập trung vào tối ưu hóa độ chính xác dự báo dựa trên các thành phần chính (PCA).

## 1. Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style="whitegrid")
print('✅ Import OK')

## 2. Tổng hợp kết quả R-squared và RMSE từ các models

Dựa trên những kết quả đã chạy từ Các Notebook 07 và 08, ta ghi nhận hiệu suất mô hình như sau:

In [ ]:
# Kết quả ghi nhận từ các Notebook trước (04, 05, 07, 08)
results = {
    'Model': ['Fixed Effects (FEM)', 'Random Effects (REM)', 'PPML', 'Spatial Econ (SLX)', 'Random Forest', 'XGBoost', 'LightGBM'],
    'Type': ['Econometrics', 'Econometrics', 'Econometrics', 'Econometrics', 'Machine Learning', 'Machine Learning', 'Machine Learning'],
    'R_squared': [0.4610, 0.1709, 0.8294, 0.5715, 0.8089, 0.7989, 0.8064],
    'RMSE': [np.nan, np.nan, np.nan, np.nan, 0.9301, 0.9540, 0.9361],
    'Main_Features': ['Gravity Vars + FE', 'Gravity Vars + RE', 'Gravity Vars', '13 Vars + 9 Lags', '6 Principal Comps', '6 Principal Comps', '6 Principal Comps']
}

df_results = pd.DataFrame(results)
# Xuất kết quả ra file csv như yêu cầu
df_results.to_csv('Table2_Gravity_Results.csv', index=False)
print('Đã xuất file Table2_Gravity_Results.csv thành công!')
df_results

## 3. Trực quan hóa so sánh ($R^2$ và RMSE)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Sử dụng biểu đồ cột đứng (Vertical Bar chart)
barplot = sns.barplot(
    data=df_results, 
    y='R_squared', 
    x='Model', 
    hue='Type', 
    dodge=False, 
    palette={'Econometrics': '#e74c3c', 'Machine Learning': '#2ecc71'}
)

# Thêm text trên các cột
for p in barplot.patches:
    height = p.get_height()
    if height > 0:
        plt.text(p.get_x() + p.get_width() / 2,
                 height + 0.02,
                 f'{height:.4f}',
                 ha='center',
                 va='bottom',
                 fontweight='bold')

ax.set_ylim(0, 1.0)
ax.set_ylabel('$R^2$ Score (Mức độ giải thích sự phân tán của dữ liệu)', fontsize=12)
ax.set_xlabel('Mô hình (Models)', fontsize=12)
ax.set_title('So sánh hệ số $R^2$ (hoặc Pseudo $R^2$) giữa các phương pháp tiếp cận', fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Phân loại mô hình', loc='upper left')

plt.tight_layout()
plt.show()

## 3.1. So sánh sai số RMSE (Chỉ cho các mô hình Machine Learning)

In [ ]:
# Lọc các model có tính RMSE (Machine Learning)
df_rmse = df_results.dropna(subset=['RMSE'])

fig, ax = plt.subplots(figsize=(8, 5))

barplot2 = sns.barplot(
    data=df_rmse, 
    x='Model', 
    y='RMSE', 
    palette='Blues_d'
)

# Thêm text trên các cột
for p in barplot2.patches:
    height = p.get_height()
    if height > 0:
        plt.text(p.get_x() + p.get_width() / 2,
                 height + 0.01,
                 f'{height:.4f}',
                 ha='center',
                 va='bottom',
                 fontweight='bold')

ax.set_ylim(0, 1.2)
ax.set_ylabel('RMSE (Sai số toàn phương trung bình)', fontsize=12)
ax.set_xlabel('Mô hình Machine Learning', fontsize=12)
ax.set_title('So sánh sai số RMSE của các thuật toán Machine Learning', fontsize=14, pad=15)

plt.tight_layout()
plt.show()

## 4. Phân tích & Đánh giá kết quả

### Khác biệt về $R^2$:
- **Kiểm định các mô hình cơ sở phân mảng (FEM / REM):** $R^2$ ở mức thấp đến trung bình (Khoảng 0.17 với REM và 0.46 với FEM) cho thấy hiện tượng di cư vẫn còn nhiều phương sai chưa thể giải thích hết nếu chỉ cố định không gian.
- **Mô hình PPML:** PPML đem lại Pseudo $R^2$ cao ($>0.82$) chứng minh việc sử dụng Poisson khắc phục rất tốt tính phi tuyến tính và phương sai sai số của di cư dạng khoảng cách. Tuy nhiên, nó không trực tiếp báo cáo được hiệu ứng lan tỏa không gian.
- **Mô hình SDM / SLX (Kinh tế lượng Không gian):** $R^2 \approx 0.57$. Nhìn chung, đối với dữ liệu mảng chéo lớn (panel dyadic data), mức giải thích 57% qua các biến tuyến tính có tính đến yếu tố không gian (lan tỏa/spillover) là chấp nhận được, cung cấp các insight ý nghĩa về các cực hút và đẩy.
- **Mô hình Machine Learning (RF, XGB, LGBM):** $R^2 \approx 0.80 - 0.81$. Khả năng học hỏi các đường cong phi tuyến (non-linear) và tương tác phức tạp giữa các thành phần. Sự ổn định và RMSE nhỏ (chỉ ~0.93) chứng minh được năng lực dự báo thực tiễn tốt.

### Sự Đánh Đổi (Trade-off) giữa Khả năng Diễn giải và Độ chính xác:
1. **Econometrics (FEM/REM, PPML, SDM/SLX):** Cung cấp *Insights*. Mô hình chỉ ra rõ ràng các tác động biên, mức độ white-box là 100%. Đây là thứ mà ML không thể làm được, rất quan trọng trong việc xây dựng chính sách (Policy Making).
2. **Machine Learning:** Cung cấp *Predictions*. Mặc dù phần lớn là black-box, nhưng tính chất tối thiểu hóa sai số dự báo (RMSE thấp, $R^2$ cực lớn) biến ML thành mũi nhọn trong việc xây dựng hệ thống quản trị rủi ro hoặc dự đoán ngắn/trung hạn.

### Kết luận chung của Dự án:
Dự án đã sử dụng một workflow toàn diện: từ **kiểm định mô hình cơ sở** (FEM, REM), đến các **mô hình lực hấp dẫn chính thống** (PPML), tiếp tục khai thác độc lập **yếu tố không gian** (SLX, SDM), và tận dụng **khoa học dữ liệu ML** để thu hẹp khoảng trống dự báo. Điều này cho phép trả lời toàn diện về cách thức quy hoạch và dự báo dòng di cư toàn cầu.